In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 109.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0


Model page: https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct

In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': "I am Qwen, an artificial intelligence developed by Alibaba Cloud. I am designed to assist with various tasks such as language translation, knowledge discovery, and data analysis. I'm here to provide information, answer questions, and help people find the answers they need. Is there anything specific you would like me to help you with today?"}]}]

In [3]:
# pip install -U transformers accelerate
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

I am Qwen, an artificial intelligence developed by Alibaba Cloud. I was created to assist users in various tasks and provide information on different topics. My primary function is to help answer questions and offer explanations to the best of my abilities based on my programming and knowledge. I'm here to support you with any questions or information you might need. How can I assist you today?<|im_end|>


In [4]:

messages = [
    {
        "role": "user",
        "content": "Write a short paragraph about the Moon in simple English."
    }
]

result = pipe(
    messages,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.7
)

print(result[0]["generated_text"][-1]["content"])


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The moon is an artificial satellite orbiting Earth. It is a natural celestial body that exists beyond our planet and has been observed since ancient times. The moon's surface is covered with craters, mountains, valleys, and plains, reflecting its geological history. Its phases change due to the gravitational pull of the Earth, making it one of the most studied objects in astronomy. Despite its small size compared to other planets, the moon plays a crucial role in the dynamics of the solar system, influencing the tides, shaping the Earth's atmosphere, and affecting the orbits of other celestial bodies. Its presence has


In [5]:

!pip install -q gradio


In [6]:

import gradio as gr

def generate_text(prompt, style, length):
    if not prompt.strip():
        return "⚠️ Please enter a topic or instruction."

    instructions = {
        "Paragraph": "Write a clear, informative paragraph about this topic.",
        "Story": "Write an interesting, creative short story based on this idea.",
        "Essay": "Write a well-organized short essay about this topic.",
        "Email": "Write a professional email based on this request.",
        "Social Media Post": "Write an engaging social media post about this topic."
    }

    messages = [
        {
            "role": "system",
            "content": "You are a helpful writing assistant. Follow the user's instructions and write clear, relevant text."
        },
        {
            "role": "user",
            "content": f"{instructions[style]}\n\nTopic or instructions: {prompt}"
        }
    ]

    result = pipe(
        messages,
        max_new_tokens=int(length),
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    return result[0]["generated_text"][-1]["content"].strip()


custom_css = """
.gradio-container {
    max-width: 1000px !important;
    margin: auto !important;
}
#main-title {
    text-align: center;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as app:
    gr.Markdown(
        "# ✨ AI Text Studio\n"
        "### Turn your ideas into well-written content",
        elem_id="main-title"
    )

    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(
                label="💡 Your Topic or Prompt",
                placeholder="Example: Write about the importance of AI in education...",
                lines=6
            )

            style = gr.Dropdown(
                choices=[
                    "Paragraph",
                    "Story",
                    "Essay",
                    "Email",
                    "Social Media Post"
                ],
                value="Paragraph",
                label="✍️ Writing Style"
            )

            length = gr.Radio(
                choices=[
                    ("Short", 80),
                    ("Medium", 150),
                    ("Long", 250)
                ],
                value=150,
                label="📏 Output Length (maximum new tokens)"
            )

            with gr.Row():
                generate_btn = gr.Button(
                    "✨ Generate Text",
                    variant="primary"
                )
                clear_btn = gr.ClearButton(
                    [prompt, style, length],
                    value="🗑️ Clear"
                )

        with gr.Column(scale=3):
            output = gr.Textbox(
                label="📝 Generated Content",
                lines=17,
                placeholder="Your generated content will appear here..."
            )

    generate_btn.click(
        fn=generate_text,
        inputs=[prompt, style, length],
        outputs=output
    )

    gr.Markdown(
        "---\n"
        "💡 **Tip:** Be specific with your prompt for more relevant results.\n\n"
        "Built with 🤗 Hugging Face, Qwen and Gradio."
    )

app.launch(share=True)


/tmp/ipykernel_549/2517935697.py:47: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab0854c3cd423fe4e2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
